In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
!pip install pillow tifffile


In [10]:
!pip install rasterio
import os
import cv2
import rasterio
import numpy as np

root_path = "/content/drive/MyDrive/project_Landset08"
input_path = f"{root_path}/subset-600"
output_hr = f"{root_path}/final_dataset/HR_Thermal"
output_lr = f"{root_path}/final_dataset/LR_Thermal"
output_opt = f"{root_path}/final_dataset/HR_Optical"

os.makedirs(output_hr, exist_ok=True)
os.makedirs(output_lr, exist_ok=True)
os.makedirs(output_opt, exist_ok=True)

ModuleNotFoundError: No module named 'rasterio'

In [11]:
!pip install rasterio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 90.2 MB/s eta 0:00:00


In [12]:
import os
import cv2
import rasterio
import numpy as np

root_path = "/content/drive/MyDrive/project_Landset08"
input_path = f"{root_path}/subset-600"
output_hr = f"{root_path}/final_dataset/HR_Thermal"
output_lr = f"{root_path}/final_dataset/LR_Thermal"
output_opt = f"{root_path}/final_dataset/HR_Optical"

os.makedirs(output_hr, exist_ok=True)
os.makedirs(output_lr, exist_ok=True)
os.makedirs(output_opt, exist_ok=True)


In [13]:
def extract_bands(allbands_path):
    with rasterio.open(allbands_path) as src:
        bands = src.read()
        # rasterio read order:
        # index 0 = Band1, index1 = Band2, ... index9 = Band10
        blue = bands[1]     # Band 2
        green = bands[2]    # Band 3
        red = bands[3]      # Band 4
        thermal = bands[9]  # Band 10 (TIR1)

    rgb = np.stack([red, green, blue], axis=-1)
    return rgb, thermal


In [14]:
def create_lr_image(hr_img):
    h, w = hr_img.shape
    lr = cv2.resize(hr_img, (w//4, h//4), interpolation=cv2.INTER_AREA)
    return lr


In [15]:
from tqdm import tqdm
import numpy as np
import os

folders = sorted(os.listdir(input_path))

for folder in tqdm(folders):
    folder_path = os.path.join(input_path, folder)

    # Check for the actual subfolder (e.g., LC08_..._...)
    # Assuming there's only one such subfolder per numbered folder
    sub_folders = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]

    if not sub_folders:
        print(f"No actual data subfolder found in: {folder_path}")
        continue

    # Use the first found subfolder for the allbands.tif path
    actual_data_folder = sub_folders[0]
    tif_path = os.path.join(folder_path, actual_data_folder, "allbands.tif")

    # Skip if file doesn't exist
    if not os.path.exists(tif_path):
        print("Missing:", tif_path)
        continue

    # Extract bands
    rgb, thermal_hr = extract_bands(tif_path)
    thermal_lr = create_lr_image(thermal_hr)

    # Save HR Thermal
    np.save(f"{output_hr}/{folder}_HR.npy", thermal_hr)

    # Save LR Thermal
    np.save(f"{output_lr}/{folder}_LR.npy", thermal_lr)

    # Save Optical RGB
    np.save(f"{output_opt}/{folder}_RGB.npy", rgb)

100%|██████████| 578/578 [00:00<00:00, 5247.35it/s]

Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000000/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000002/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000003/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000004/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000006/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000007/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000008/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000010/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000011/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000012/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000015/allbands.tif
Missing: /content/drive/MyDrive/project_Landset08/subset-600/0000016/allbands.tif
Missing: /conten

Let's first check the contents of your `input_path` (`/content/drive/MyDrive/project_Landset08/subset-600`) to see the subfolders and their contents. This will help us understand why `allbands.tif` files are being reported as missing.

In [16]:
import os

# List the contents of the input_path directory
print(f"Contents of {input_path}:")
for item in os.listdir(input_path):
    print(f"- {item}")

# If there are subfolders, let's check one of them for 'allbands.tif'
# We'll use the first folder found if available
folders = sorted(os.listdir(input_path))
if folders:
    sample_folder = folders[0]
    sample_folder_path = os.path.join(input_path, sample_folder)
    print(f"\nContents of a sample subfolder ({sample_folder_path}):")
    if os.path.isdir(sample_folder_path):
        for item in os.listdir(sample_folder_path):
            print(f"- {item}")
    else:
        print(f"'{sample_folder}' is not a directory.")
else:
    print("No subfolders found in the input directory.")

Contents of /content/drive/MyDrive/project_Landset08/subset-600:
- 0016072
- 0016081
- 0016079
- 0016080
- 0016078
- 0016076
- 0016074
- 0016071
- 0016069
- 0016070
- 0016067
- 0016064
- 0016065
- 0016060
- 0016062
- 0016058
- 0016061
- 0016068
- 0016059
- 0016063
- 0016047
- 0016050
- 0016053
- 0016055
- 0016049
- 0016057
- 0016056
- 0016048
- 0016052
- 0016051
- 0016040
- 0000650
- 0000648
- 0016039
- 0000647
- 0016043
- 0016045
- 0016044
- 0016042
- 0000649
- 0000639
- 0000643
- 0000642
- 0000638
- 0000641
- 0000644
- 0000637
- 0000645
- 0000635
- 0000636
- 0000627
- 0000630
- 0000629
- 0000631
- 0000625
- 0000632
- 0000633
- 0000624
- 0000628
- 0000623
- 0000622
- 0000621
- 0000618
- 0000616
- 0000610
- 0000614
- 0000619
- 0000613
- 0000611
- 0000608
- 0000606
- 0000596
- 0000605
- 0000603
- 0000601
- 0000598
- 0000599
- 0000607
- 0000604
- 0000595
- 0000584
- 0000589
- 0000581
- 0000593
- 0000582
- 0000587
- 0000590
- 0000594
- 0000583
- 0000588
- 0000574
- 0000569
- 0000573
- 000

In [17]:
from tqdm import tqdm
import numpy as np
import os
import rasterio
import cv2

# Paths
input_path = "/content/drive/MyDrive/project_Landset08/subset-600"
output_hr = "/content/drive/MyDrive/project_Landset08/HR"
output_lr = "/content/drive/MyDrive/project_Landset08/LR"
output_opt = "/content/drive/MyDrive/project_Landset08/RGB"

# Extract required bands from all_bands.tif
def extract_bands(tif_path):
    with rasterio.open(tif_path) as src:
        b2 = src.read(2)   # Blue
        b3 = src.read(3)   # Green
        b4 = src.read(4)   # Red
        b10 = src.read(10) # Thermal infrared

    rgb = np.stack([b4, b3, b2], axis=-1)   # (H, W, 3)
    return rgb, b10   # HR thermal

# Downsample HR thermal → LR thermal
def create_lr_image(thermal_hr):
    # Bilinear downsampling to 1/4 resolution (example)
    lr = cv2.resize(
        thermal_hr,
        (thermal_hr.shape[1] // 4, thermal_hr.shape[0] // 4),
        interpolation=cv2.INTER_LINEAR
    )
    return lr

# ---------------------------------------------------------

folders = sorted(os.listdir(input_path))

for folder in tqdm(folders):
    folder_path = os.path.join(input_path, folder)

    # List items inside 0000000, 0000001, ...
    sub = [
        d for d in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, d))
    ]

    if not sub:
        print("⚠ No LC08 folder found in:", folder_path)
        continue

    lc08_folder = sub[0]   # e.g., LC08_045030_20190814
    tif_path = os.path.join(folder_path, lc08_folder, "all_bands.tif")

    if not os.path.exists(tif_path):
        print("⚠ Missing file:", tif_path)
        continue

    # Extract bands
    rgb, thermal_hr = extract_bands(tif_path)
    thermal_lr = create_lr_image(thermal_hr)

    # Save files
    np.save(f"{output_hr}/{folder}_HR.npy", thermal_hr)
    np.save(f"{output_lr}/{folder}_LR.npy", thermal_lr)
    np.save(f"{output_opt}/{folder}_RGB.npy", rgb)


  0%|          | 0/578 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/project_Landset08/HR/0000000_HR.npy'

In [18]:
import rasterio

path = "/content/drive/MyDrive/project_Landset08/subset-600/0000000/LC08_045030_20190814/all_bands.tif"

with rasterio.open(path) as src:
    print("Total Bands:", src.count)
    print("Width:", src.width)
    print("Height:", src.height)
    print("CRS:", src.crs)
    print("Data Type:", src.dtypes)


Total Bands: 11
Width: 264
Height: 264
CRS: EPSG:4326
Data Type: ('uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8', 'uint8')


In [19]:
from tqdm import tqdm
import numpy as np
import os
import rasterio
import cv2

# === Paths ===
input_path = "/content/drive/MyDrive/project_Landset08/subset-600"
output_hr = "/content/drive/MyDrive/project_Landset08/final_dataset/HR_Thermal"
output_lr = "/content/drive/MyDrive/project_Landset08/final_dataset/LR_Thermal"
output_rgb = "/content/drive/MyDrive/project_Landset08/final_dataset/HR_Optical"

# Ensure folders exist
os.makedirs(output_hr, exist_ok=True)
os.makedirs(output_lr, exist_ok=True)
os.makedirs(output_rgb, exist_ok=True)

def extract_bands(tif_path):
    with rasterio.open(tif_path) as src:
        bands = src.read()
        rgb = np.stack([bands[3], bands[2], bands[1]], axis=-1)  # B4, B3, B2
        thermal = bands[9]  # Band 10
        return rgb, thermal

def create_lr_image(hr):
    lr = cv2.resize(hr, (hr.shape[1]//4, hr.shape[0]//4), interpolation=cv2.INTER_AREA)
    lr_up = cv2.resize(lr, (hr.shape[1], hr.shape[0]), interpolation=cv2.INTER_CUBIC)
    return lr_up

folders = sorted(os.listdir(input_path))

for folder in tqdm(folders):
    folder_path = os.path.join(input_path, folder)

    # Find subfolder (LC08_....)
    subs = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]
    if not subs:
        print(f"Skipping: {folder_path}")
        continue

    subfolder = subs[0]  # LC08_xxxxxxxx
    tif_path = os.path.join(folder_path, subfolder, "all_bands.tif")

    if not os.path.exists(tif_path):
        print("Missing:", tif_path)
        continue

    # Extract
    rgb, thermal_hr = extract_bands(tif_path)
    thermal_lr = create_lr_image(thermal_hr)

    # Save
    np.save(f"{output_hr}/{folder}_HR.npy", thermal_hr)
    np.save(f"{output_lr}/{folder}_LR.npy", thermal_lr)
    np.save(f"{output_rgb}/{folder}_RGB.npy", rgb)


100%|██████████| 578/578 [02:26<00:00,  3.95it/s]


In [20]:
import os
import random

dataset_path = "/content/drive/MyDrive/project_Landset08/final_dataset/HR_Thermal"

# Get all samples (e.g., 0000000_HR.npy)
all_files = sorted([f.replace("_HR.npy", "") for f in os.listdir(dataset_path) if f.endswith("_HR.npy")])

random.shuffle(all_files)

total = len(all_files)
train_split = int(0.8 * total)
val_split = int(0.9 * total)

train = all_files[:train_split]
val = all_files[train_split:val_split]
test = all_files[val_split:]

split_dir = "/content/drive/MyDrive/project_Landset08/final_dataset/splits"
os.makedirs(split_dir, exist_ok=True)

with open(f"{split_dir}/train.txt", "w") as f:
    f.write("\n".join(train))

with open(f"{split_dir}/val.txt", "w") as f:
    f.write("\n".join(val))

with open(f"{split_dir}/test.txt", "w") as f:
    f.write("\n".join(test))

print("Splits created successfully!")
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")


Splits created successfully!
Train: 462, Val: 58, Test: 58
